In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import yfinance as yf
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

In [2]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import yfinance as yf

class ForexEnv(gym.Env):
    def __init__(self, df, initial_balance=1000):
        super(ForexEnv, self).__init__()
        self.df = df.reset_index(drop=True)
        self.initial_balance = float(initial_balance)
        self.balance = float(initial_balance)
        self.usd = 0.0
        self.current_step = 0

        # Observation = [prix, balance, usd]
        self.observation_space = spaces.Box(
            low=0, high=np.inf, shape=(3,), dtype=np.float32
        )
        # Actions : 0=Hold, 1=Buy, 2=Sell
        self.action_space = spaces.Discrete(3)

    def _get_obs(self):
        price = float(self.df.loc[self.current_step, "Close"])
        # force conversion float sur tous les éléments
        return np.array([
            float(price),
            float(self.balance),
            float(self.usd)
        ], dtype=np.float32)

    def step(self, action):
        price = float(self.df.loc[self.current_step, "Close"])
        reward = 0.0

        if action == 1 and self.balance > 0:  # BUY USD
            self.usd += self.balance / price
            self.balance = 0.0
        elif action == 2 and self.usd > 0:  # SELL USD
            self.balance += self.usd * price
            self.usd = 0.0
            reward = self.balance - self.initial_balance

        self.current_step += 1
        terminated = self.current_step >= len(self.df) - 1
        truncated = False
        obs = self._get_obs()
        info = {}

        return obs, float(reward), terminated, truncated, info

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.balance = float(self.initial_balance)
        self.usd = 0.0
        self.current_step = 0
        obs = self._get_obs()
        info = {}
        return obs, info


In [3]:
# ============================================================
# 2️⃣ CHARGEMENT DES DONNÉES EUR/USD
# ============================================================

df = yf.download("EURUSD=X", start="2020-01-01", end="2023-01-01", interval="1d").dropna().reset_index(drop=True)

env = ForexEnv(df)
env = Monitor(env)
env = DummyVecEnv([lambda: env])

model = PPO("MlpPolicy", env, verbose=1, tensorboard_log="./logs/forex")
model.learn(total_timesteps=100000)


C:\Users\Aurelien\AppData\Local\Temp\ipykernel_45792\1614776742.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("EURUSD=X", start="2020-01-01", end="2023-01-01", interval="1d").dropna().reset_index(drop=True)
[*********************100%***********************]  1 of 1 completed


Using cpu device
Logging to ./logs/forex\PPO_2


C:\Users\Aurelien\AppData\Local\Temp\ipykernel_45792\3079427301.py:24: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  price = float(self.df.loc[self.current_step, "Close"])
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_45792\3079427301.py:33: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  price = float(self.df.loc[self.current_step, "Close"])


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 782      |
|    ep_rew_mean     | 735      |
| time/              |          |
|    fps             | 896      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 782         |
|    ep_rew_mean          | -837        |
| time/                   |             |
|    fps                  | 700         |
|    iterations           | 2           |
|    time_elapsed         | 5           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.007812959 |
|    clip_fraction        | 0.0411      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | -5.04e-05   |
|    learning_rate        | 0.

In [4]:
# ============================================================
# 3️⃣ AGENT PPO
# ============================================================
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    tensorboard_log="./logs/forex",
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    ent_coef=0.01
)

Using cpu device


In [5]:
# ============================================================
# 4️⃣ APPRENTISSAGE
# ============================================================
model.learn(total_timesteps=100000)

Logging to ./logs/forex\PPO_3


C:\Users\Aurelien\AppData\Local\Temp\ipykernel_45792\3079427301.py:24: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  price = float(self.df.loc[self.current_step, "Close"])
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_45792\3079427301.py:33: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  price = float(self.df.loc[self.current_step, "Close"])


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 782      |
|    ep_rew_mean     | -128     |
| time/              |          |
|    fps             | 887      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 782         |
|    ep_rew_mean          | 985         |
| time/                   |             |
|    fps                  | 734         |
|    iterations           | 2           |
|    time_elapsed         | 5           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009035056 |
|    clip_fraction        | 0.024       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | -0.000339   |
|    learning_rate        | 0.

In [ ]:
# ============================================================
# 5️⃣ SAUVEGARDE DU MODÈLE
# ============================================================
model.save("models/ppo_forex_model")

print("✅ Entraînement terminé. Modèle sauvegardé dans 'ppo_forex_model'.")

✅ Entraînement terminé. Modèle sauvegardé dans 'ppo_forex_model'.
